In [10]:
import pandas as pd
import numpy as np
from anjana.anonymity import (
    k_anonymity_inner,
    k_anonymity,
    l_diversity,
    t_closeness,
    alpha_k_anonymity,
)
import masking_effects_on_xai_techniques.hierarchies as hi

In [11]:
path = "../data/usa_house/"

In [30]:
data = pd.read_csv(path + "data.csv")
len_data = len(data)

# Clean the data
data.dropna(inplace=True)
data = data.drop("Address", axis=1)

len_clean_data = len(data)
print(f"Dropped {len_data - len_clean_data} rows")
df = data
data.to_csv(path + "clean.csv", index=False)

Dropped 0 rows


In [27]:
categories = data.columns.to_list()

hierarchies = dict()
for cat in categories:
    hierarchies[cat] = hi.generate_qcut_hierarchy(data[cat], 7)
    hi.save_hierarchy(
        hierarchies[cat], f"../hierarchies/usa_house/{cat}.csv", sort=False
    )
    hierarchies[cat] = dict(
        pd.read_csv(f"../hierarchies/usa_house/{cat}.csv", header=None)
    )

/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/p

In [19]:
# Check with mina what should be identified as quai identifiers
quasi_ident = list(data.columns)

# quasi_ident.remove('capital-gain')
# quasi_ident.remove('capital-loss')
# quasi_ident.remove('hours-per-week')

ident = []  # Making race quasi identifier make k much larger
sens_att = "Price"

In [21]:
all_counts_and_features = []
for feat, items in hierarchies.items():
    for item in items.values():
        length = len(item.unique())
        all_counts_and_features.append((length, feat))
sorted_data = sorted(all_counts_and_features, key=lambda x: x[0], reverse=True)
ordered_features = [feat for count, feat in sorted_data]

for feat, items in hierarchies.items():
    lengths = [f"{len(item.unique()):>4}" for item in items.values()]
    out = ", ".join(lengths)
    print(f"{feat:<15}: {out}")

print("Order in which features will be generalized:")
print(ordered_features)

Avg. Area Income: 5000,    6,    5,    4,    3,    2,    1
Avg. Area House Age: 5000,    6,    5,    4,    3,    2,    1
Avg. Area Number of Rooms: 5000,    6,    5,    4,    3,    2,    1
Avg. Area Number of Bedrooms:  255,    6,    5,    4,    3,    2,    1
Area Population: 5000,    6,    5,    4,    3,    2,    1
Price          : 5000,    6,    5,    4,    3,    2,    1
Order in which features will be generalized:
['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Area Population', 'Price', 'Avg. Area Number of Bedrooms', 'Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Avg. Area Number of Bedrooms', 'Area Population', 'Price', 'Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Avg. Area Number of Bedrooms', 'Area Population', 'Price', 'Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Avg. Area Number of Bedrooms', 'Area Population', 'Price', 'Avg. Area Income', 'Avg. Area House Age', 'Avg. A

In [28]:
# Check with mina if this k is representative of used k in litterature
k = 10
supp_level = 20  # Select the suppression limit allowed
anon_df, supp_n, hiear = k_anonymity_inner(
    data, ident, quasi_ident, k, supp_level, hierarchies
)
max_supp_n = int(round(len(data) * supp_level / 100, 0))
print(f"Max rows that can be suppressed: {max_supp_n}")
print(f"Rows suppressed                : {supp_n}")
print(f"% of allowed rows suppressed   : {round(supp_n / max_supp_n * 100, 1)}%")
print(f"Generalization level           : {sum(hiear.values())}")
hiear

Max rows that can be suppressed: 1000
Rows suppressed                : 831
% of allowed rows suppressed   : 83.1%
Generalization level           : 25


{'Avg. Area Income': 5,
 'Avg. Area House Age': 4,
 'Avg. Area Number of Rooms': 4,
 'Avg. Area Number of Bedrooms': 4,
 'Area Population': 4,
 'Price': 4}

In [31]:
for t in np.linspace(0.1, 1.0, 10):
    t = round(t, 1)
    print(f"---- {t} ----")
    if df.empty:
        print("Skipping")
        continue
    t_closeness(
        data, ident, quasi_ident, sens_att, k, t, supp_level, hierarchies
    ).to_csv(f"../data/t_closeness/{t}.csv", index=False)

---- 0.1 ----


OSError: Cannot save file into a non-existent directory: '../data/t_closeness'

In [ ]:
for alpha in np.linspace(0.1, 1.0, 10):
    alpha = round(alpha, 1)
    print(f"---- {alpha} ----")
    df = alpha_k_anonymity(
        data, ident, quasi_ident, sens_att, k, alpha, supp_level, hierarchies
    )
    if df.empty:
        print("Skipping")
        continue
    df.to_csv(f"../data/alpha_k_anonymity/{alpha}.csv", index=False)

In [ ]:
for l in range(1, 11):
    print(f"---- {l} ----")
    df = l_diversity(data, ident, quasi_ident, sens_att, k, l, supp_level, hierarchies)
    if df.empty:
        print("Skipping")
        continue
    df.to_csv(f"../data/l_diversity/{l}.csv", index=False)